# Fit BERTopic models (the producer for `results_semi/`)

This is the step you run **before** any Track 2 analysis. It fits the topic
models and writes their assignments into `results/track2_bertopic/results_semi/`,
which every analysis and case-study notebook then reads through the merger.

It produces one folder per model:

- `unsupervised/` — fully unsupervised BERTopic (no label guidance)
- `<label_target>_<weight>/` — semi-supervised runs, e.g. `main_0.3`,
  `report_accident_0.25`, `all_0.2`

Each folder gets `document_topics.csv`, `topic_info.csv` and `topic_words.json`,
the format the analysis notebooks expect.

On the synthetic data this fits real models on fake narratives, so the topics
are meaningless by construction. The `make_synthetic_data.py` stand-ins already
write the same folders, so the analysis notebooks run even without this step;
run this to regenerate them properly on the real VD data.

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path

# put src/ on the path
_p = os.getcwd()
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, os.path.join(_p, "src")); break
    _p = os.path.dirname(_p)

import numpy as np
import pandas as pd

from data_load import load_data_for_modeling
from embedding import load_sentence_embedding_model, generate_embeddings
from stop_words import get_stop_words
from bertopic_components import BERTopicConfig, run_bertopic, set_seed, SEED
from config import BASE_DATA_FOLDER, RESULTS_SEMI_DIR

set_seed(SEED)
RESULTS_SEMI_DIR = Path(RESULTS_SEMI_DIR)

## 2. Load narratives, stop words, and embeddings

In [ ]:
data = load_data_for_modeling(data_folder=BASE_DATA_FOLDER, encode_labels=False, verbose=True)
df = data["df"].reset_index(drop=True)
docs = data["docs"]

stop_words = get_stop_words()

# Pre-compute embeddings once and reuse across every model fit.
embedding_model = load_sentence_embedding_model("KennethTM/MiniLM-L6-danish-encoder")
embeddings = generate_embeddings(docs, embedding_model, verbose=True)

## 3. Final hyperparameters and the save helper

In [ ]:
# Final settings selected in the hyperparameter exploration.
# The synthetic sample is intentionally smaller than the real thesis data, so
# use conservative cluster/vectorizer settings when running on it.
IS_SYNTHETIC = "synthetic" in str(BASE_DATA_FOLDER)

FINAL_PARAMS = dict(
    n_neighbors=30 if IS_SYNTHETIC else 65,
    n_components=5 if IS_SYNTHETIC else 10,
    min_dist=0.01,
    min_cluster_size=25 if IS_SYNTHETIC else 900,
    min_samples=5 if IS_SYNTHETIC else 10,
    min_df=1 if IS_SYNTHETIC else 2,
    ngram_range=(1, 2),
    nr_topics=None if IS_SYNTHETIC else "auto",
    top_n_words=20,
)

def save_results(model, topics, probs, out_dir):
    """Write document_topics.csv, topic_info.csv and topic_words.json."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if probs is None:
        probs = [None] * len(topics)

    pd.DataFrame({
        "document_index": np.arange(len(topics)),
        "assigned_topic": topics,
        "assigned_topic_probability": probs,
    }).to_csv(out_dir / "document_topics.csv", index=False)

    model.get_topic_info().to_csv(out_dir / "topic_info.csv", index=False)

    topic_words = {
        str(t): [[w, float(s)] for w, s in model.get_topic(t)]
        for t in model.get_topics() if t != -1
    }
    with open(out_dir / "topic_words.json", "w", encoding="utf-8") as f:
        json.dump(topic_words, f, ensure_ascii=False, indent=2)

    print(f"  saved -> {out_dir}")


## 4. Fully unsupervised model

Standard BERTopic with no label guidance, saved to `results_semi/unsupervised/`.

In [ ]:
set_seed(SEED)
cfg = BERTopicConfig(mode="unsupervised", stop_words=stop_words, **FINAL_PARAMS)
model, topics, probs = run_bertopic(cfg, docs, embeddings, embedding_model)
save_results(model, topics, probs, RESULTS_SEMI_DIR / "unsupervised")

## 5. Semi-supervised models

Each run biases the embedding geometry toward a structured label target via
UMAP's `target_weight`, then saves to `results_semi/<label_target>_<weight>/`.

The label columns below are the integer class labels passed to UMAP. `main`
uses `main_situation_class` directly; adjust the `report_accident` and `all`
targets to your actual label definitions if they differ.

In [ ]:
# Integer label arrays, one value per document.
LABEL_TARGETS = {
    "main":            df["main_situation_class"].astype(int).tolist(),
    "report_accident": pd.factorize(df["report_category"])[0].tolist(),
    "all":             pd.factorize(df["encoded_accident_situation"])[0].tolist(),
}

# (label_target, target_weight) -> folder name f"{label_target}_{weight}"
SEMI_CONFIGS = [
    ("main", 0.3),
    ("report_accident", 0.25),
    ("all", 0.2),
]

for label_target, weight in SEMI_CONFIGS:
    print(f"Fitting semi-supervised: {label_target} (weight {weight})")
    set_seed(SEED)
    cfg = BERTopicConfig(
        mode="semi_supervised",
        labels=LABEL_TARGETS[label_target],
        target_weight=weight,
        stop_words=stop_words,
        **FINAL_PARAMS,
    )
    model, topics, probs = run_bertopic(cfg, docs, embeddings, embedding_model)
    save_results(model, topics, probs, RESULTS_SEMI_DIR / f"{label_target}_{weight}")